# Practice Lab 4.6 - Ragas Evaluation as a CI Gate (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 4 - Lesson 4.6 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-4.6.html` (8 graded
exercises, 40 points, pass at 28/40). Each exercise below repeats the prompt and success
criteria, then shows a runnable worked solution. Run it top to bottom.

- Exercises 1 and 6 touch the Ragas 0.4.x live judge. **No key needed to run:** those cells
  branch on `HAS_KEY` and fall back to a deterministic offline scorecard, and the keyless guard
  in the next cell mocks the anthropic / openai clients so every cell executes end to end. Set
  `OPENAI_API_KEY` (or `ANTHROPIC_API_KEY`) to reproduce the live-judged numbers.
- Exercises 3, 5, 7, and 8 run fully offline on inline data (numpy / stdlib only).
- Exercise 4 ships a GitHub Actions workflow (YAML reference, not an executable cell).
- Production context: AcmeCorp's RAG for an EU bank governed by GDPR + the EU AI Act, judged in
  `us-east1`, gated on the canonical four-metric floor (faithfulness / answer-relevancy 0.85,
  context precision / recall 0.80), hardening to faithfulness 0.90 for the enterprise renewal.


In [ ]:
!pip install --quiet ragas openai anthropic numpy langchain-openai

In [ ]:
# --- Keyless runnable guard (auto-inserted from the lesson HTML) --------------
# Learners WITH an ANTHROPIC_API_KEY / OPENAI_API_KEY hit the live model.
# WITHOUT a key, the common clients are mocked so this notebook still runs
# top-to-bottom with deterministic canned responses (no network, no cost).
import os
_HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
if not (_HAS_ANTHROPIC or _HAS_OPENAI):
    print("No API key set -> KEYLESS MOCK mode (canned responses; set a key to go live).")
    try:
        import anthropic  # type: ignore
        class _MsgBlock:
            def __init__(self, text): self.text = text; self.type = "text"
        class _Msg:
            def __init__(self, text):
                self.content = [_MsgBlock(text)]
                self.stop_reason = "end_turn"
                self.usage = type("U", (), {"input_tokens": 0, "output_tokens": 0})()
        class _Messages:
            def create(self, **kw):
                return _Msg("[keyless-mock] set ANTHROPIC_API_KEY for a live response.")
        class _Anthropic:
            def __init__(self, *a, **k): self.messages = _Messages()
        anthropic.Anthropic = _Anthropic
        if hasattr(anthropic, "AsyncAnthropic"): anthropic.AsyncAnthropic = _Anthropic
    except Exception:
        pass
    try:
        import openai  # type: ignore
        class _OAMsg:
            def __init__(self, text): self.content = text
        class _OAChoice:
            def __init__(self, text): self.message = _OAMsg(text)
        class _OAResp:
            def __init__(self, text): self.choices = [_OAChoice(text)]
        class _Completions:
            def create(self, **kw):
                return _OAResp("[keyless-mock] set OPENAI_API_KEY for a live response.")
        class _Chat:
            def __init__(self): self.completions = _Completions()
        class _OpenAI:
            def __init__(self, *a, **k): self.chat = _Chat()
        openai.OpenAI = _OpenAI
        if hasattr(openai, "AsyncOpenAI"): openai.AsyncOpenAI = _OpenAI
    except Exception:
        pass
# ----------------------------------------------------------------------------

---
## Exercise 1 - Install ragas 0.4.x + score 3 hand-crafted rows  (Easy, 5 pts)

**Task.** Install `ragas` (0.4.x) and score a 3-row hand-crafted RAG with the four
class-based metrics - `Faithfulness`, `ResponseRelevancy`, `LLMContextPrecisionWithReference`,
`LLMContextRecall`. One row hallucinates, one is fed the wrong context, one is correct. Build an
`EvaluationDataset` of `SingleTurnSample` objects (`user_input` / `response` /
`retrieved_contexts` / `reference`), pass an `evaluator_llm`, and read `result.to_pandas()`. The
lowercase singletons and the `question/answer/contexts/ground_truth` keys are the old v0.1 API.

**Success criteria**
- A running per-row scorecard on the 0.4.x API (3 rows).
- Commentary explaining which metric flagged which failure (faithfulness -> row 1,
  context precision -> row 2).

In [ ]:
# Worked solution - 3 hand-crafted rows (correct / hallucinated / wrong-context),
# scored on the Ragas 0.4.x class-based API. Runs live with a key; deterministic offline
# scorecard otherwise. The lowercase-singleton / question-answer-contexts v0.1 API is gone.
import os
HAS_KEY = bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("ANTHROPIC_API_KEY"))

# Each row targets one failure mode on purpose.
rows = [
    {   # row 0 - correct: answer follows from context, right context retrieved
        "user_input": "What is AcmeCorp's retention window for KYC records?",
        "response": "AcmeCorp retains KYC records for 5 years after account closure (Data-Protection Standard clause 4).",
        "retrieved_contexts": ["AcmeCorp Data-Protection Standard clause 4: KYC records are retained for 5 years after account closure."],
        "reference": "Per AcmeCorp Data-Protection Standard clause 4, KYC records are retained for 5 years after account closure.",
    },
    {   # row 1 - hallucination: right context, but the answer invents 10 years
        "user_input": "What is AcmeCorp's retention window for KYC records?",
        "response": "AcmeCorp retains KYC records for 10 years after account closure.",
        "retrieved_contexts": ["AcmeCorp Data-Protection Standard clause 4: KYC records are retained for 5 years after account closure."],
        "reference": "Per AcmeCorp Data-Protection Standard clause 4, KYC records are retained for 5 years after account closure.",
    },
    {   # row 2 - wrong context: an unrelated refund chunk retrieved for a KYC question
        "user_input": "What is AcmeCorp's retention window for KYC records?",
        "response": "I could not find the KYC retention window in the retrieved policy text.",
        "retrieved_contexts": ["AcmeCorp Refund Policy Section 3.2: refund disputes are resolved within 10 working days."],
        "reference": "Per AcmeCorp Data-Protection Standard clause 4, KYC records are retained for 5 years after account closure.",
    },
]

if HAS_KEY:
    from ragas import evaluate, EvaluationDataset
    from ragas.dataset_schema import SingleTurnSample
    from ragas.metrics import (Faithfulness, ResponseRelevancy,
        LLMContextPrecisionWithReference, LLMContextRecall)
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings

    judge = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.6"))
    emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))
    dataset = EvaluationDataset(samples=[SingleTurnSample(**r) for r in rows])
    result = evaluate(dataset=dataset,
        metrics=[Faithfulness(), ResponseRelevancy(),
                 LLMContextPrecisionWithReference(), LLMContextRecall()],
        llm=judge, embeddings=emb)
    print(result)
    print(result.to_pandas())
else:
    # Deterministic offline scorecard - the shape a live judge returns for these 3 rows.
    OFFLINE = {
        "faithfulness":      [0.98, 0.30, 0.90],   # row 1 hallucination tanks faithfulness
        "answer_relevancy":  [0.96, 0.92, 0.40],
        "context_precision": [1.00, 1.00, 0.00],   # row 2 wrong-context tanks precision
        "context_recall":    [1.00, 1.00, 0.00],
    }
    notes = ["correct", "hallucinated (10y)", "wrong context"]
    print(f'{"row":>3} {"faith":>7} {"a_rel":>7} {"c_prec":>7} {"c_rec":>7}  note')
    for i in range(3):
        print(f'{i:>3} {OFFLINE["faithfulness"][i]:>7.2f} {OFFLINE["answer_relevancy"][i]:>7.2f} '
              f'{OFFLINE["context_precision"][i]:>7.2f} {OFFLINE["context_recall"][i]:>7.2f}  {notes[i]}')
    print("\nRead per-row, not the aggregate: faithfulness flags row 1, context precision flags row 2.")

---
## Exercise 2 - Build a 10-question golden set for AcmeCorp's EU-bank policy corpus  (Easy, 5 pts)

**Task.** Author a 10-question golden dataset over AcmeCorp's EU-bank policy corpus (GDPR
articles, internal AML/KYC procedures, EU AI Act obligations). Cover 4 query types: citation,
paraphrase, multi-hop (needs 2 chunks), and one non-English (French or German) query. Each row
carries `user_input`, `reference` (SME ground-truth), `reference_contexts` (gold chunks), and a
`metadata` dict with `type` + `lang`.

**Success criteria**
- 10 rows in valid JSONL.
- Ground-truth answers use formal regulatory language.
- `reference_contexts` cite specific articles / sub-clauses by ID.
- At least 1 non-English (French or German) query.

In [ ]:
# Worked solution - a 10-row golden set over AcmeCorp's EU-bank policy corpus.
# Types: citation, paraphrase, multi_hop, non_english (fr/de). Each row carries
# user_input, reference (SME ground-truth in formal regulatory language),
# reference_contexts (gold chunks cited by article/clause), metadata{type,lang}.
import json

GOLDEN = [
    {"user_input": "Which GDPR article establishes the storage-limitation principle AcmeCorp implements in clause 4?",
     "reference": "GDPR Article 5(1)(e) establishes the storage-limitation principle; AcmeCorp Data-Protection Standard clause 4 implements it as a 90-day default retention window post-purpose unless a statute requires longer.",
     "reference_contexts": ["GDPR Art. 5(1)(e): personal data kept in a form permitting identification of data subjects for no longer than necessary.",
                            "AcmeCorp Data-Protection Standard clause 4: default retention 90 days post-purpose."],
     "metadata": {"type": "citation", "lang": "en"}},
    {"user_input": "Under the EU AI Act, what obligation does Article 14 place on the high-risk AI system used in customer onboarding?",
     "reference": "EU AI Act Article 14 requires that high-risk AI systems be designed to enable effective human oversight, so the onboarding model must allow a competent person to intervene in or override its output.",
     "reference_contexts": ["EU AI Act Art. 14: high-risk AI systems shall be designed and developed to enable effective oversight by natural persons."],
     "metadata": {"type": "citation", "lang": "en"}},
    {"user_input": "Which GDPR article supplies the legal basis the bank relies on for mandatory AML screening?",
     "reference": "GDPR Article 6(1)(c) supplies the legal basis: processing necessary for compliance with a legal obligation to which the controller is subject, which covers the bank's statutory anti-money-laundering screening duty.",
     "reference_contexts": ["GDPR Art. 6(1)(c): processing necessary for compliance with a legal obligation to which the controller is subject."],
     "metadata": {"type": "citation", "lang": "en"}},
    {"user_input": "In plain terms, how long may AcmeCorp keep a customer's KYC file after the account is closed?",
     "reference": "AcmeCorp's internal AML/KYC procedure requires KYC records to be retained for five years after the business relationship ends, consistent with the applicable anti-money-laundering record-keeping obligation.",
     "reference_contexts": ["AcmeCorp AML/KYC Procedure Section 7: retain KYC records for 5 years after the end of the business relationship."],
     "metadata": {"type": "paraphrase", "lang": "en"}},
    {"user_input": "Roughly speaking, what must the bank do before it acts on an automated AML alert?",
     "reference": "Before any adverse action is taken on an automated AML alert, a compliance analyst must review and confirm it, because the EU AI Act's human-oversight requirement precludes a solely automated adverse decision.",
     "reference_contexts": ["EU AI Act Art. 14: effective human oversight of high-risk AI systems.",
                            "AcmeCorp AML/KYC Procedure Section 9: analyst confirmation required before filing a suspicious-activity report."],
     "metadata": {"type": "paraphrase", "lang": "en"}},
    {"user_input": "If a customer exercises the right to erasure, may AcmeCorp still retain their KYC records, and on what basis?",
     "reference": "Yes. GDPR Article 17(1) grants the right to erasure, but Article 17(3)(b) disapplies it where processing is necessary for compliance with a legal obligation, so AcmeCorp retains the KYC records for the five-year AML retention period notwithstanding the erasure request.",
     "reference_contexts": ["GDPR Art. 17(1): the data subject has the right to obtain erasure of personal data.",
                            "GDPR Art. 17(3)(b): erasure does not apply where processing is necessary for compliance with a legal obligation.",
                            "AcmeCorp AML/KYC Procedure Section 7: 5-year KYC retention."],
     "metadata": {"type": "multi_hop", "lang": "en"}},
    {"user_input": "When an AML model flags a customer, which two obligations - one from the GDPR, one from the EU AI Act - together govern how the bank must handle the flag?",
     "reference": "GDPR Article 22(1) restricts decisions based solely on automated processing that produce legal effects, and EU AI Act Article 14 mandates human oversight of high-risk AI; together they require a human analyst to review the flag before any adverse action.",
     "reference_contexts": ["GDPR Art. 22(1): right not to be subject to a decision based solely on automated processing producing legal effects.",
                            "EU AI Act Art. 14: human oversight of high-risk AI systems."],
     "metadata": {"type": "multi_hop", "lang": "en"}},
    {"user_input": "Quelle est la duree de conservation par defaut des donnees personnelles selon la clause 4 de la norme de protection des donnees d'AcmeCorp?",
     "reference": "Selon la clause 4 de la norme de protection des donnees d'AcmeCorp, qui met en oeuvre l'article 5 du RGPD, la duree de conservation par defaut est de 90 jours apres la finalite, sauf obligation legale imposant une duree plus longue.",
     "reference_contexts": ["AcmeCorp Data-Protection Standard clause 4: conservation par defaut de 90 jours apres la finalite (met en oeuvre l'art. 5 du RGPD)."],
     "metadata": {"type": "non_english", "lang": "fr"}},
    {"user_input": "Welcher Artikel der DSGVO regelt das Auskunftsrecht der betroffenen Person hinsichtlich ihrer bei AcmeCorp gespeicherten Daten?",
     "reference": "Artikel 15 der DSGVO regelt das Auskunftsrecht der betroffenen Person; AcmeCorp muss auf Anfrage bestaetigen, welche personenbezogenen Daten verarbeitet werden, und eine Kopie dieser Daten bereitstellen.",
     "reference_contexts": ["DSGVO Art. 15: Auskunftsrecht der betroffenen Person ueber die Verarbeitung ihrer personenbezogenen Daten."],
     "metadata": {"type": "non_english", "lang": "de"}},
    {"user_input": "What does AcmeCorp's life-insurance underwriting policy say about processing genetic data?",
     "reference": "Not in retrieved context.",
     "reference_contexts": [],
     "metadata": {"type": "out_of_corpus", "lang": "en"}},
]

with open("golden.jsonl", "w", encoding="utf-8") as f:
    for row in GOLDEN:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

from collections import Counter
types = Counter(r["metadata"]["type"] for r in GOLDEN)
non_english = [r["metadata"]["lang"] for r in GOLDEN if r["metadata"]["lang"] != "en"]
assert len(GOLDEN) == 10, "golden set must have 10 rows"
assert len(non_english) >= 1, "need at least one non-English query"
print("golden.jsonl written:", len(GOLDEN), "rows")
print("types:", dict(types))
print("non-English coverage:", non_english)

---
## Exercise 3 - pytest fixture with the Ragas threshold gate  (Medium, 5 pts)

**Task.** Wire the Ragas gate into a single `test_ragas_gate` function that asserts the
canonical CI-gate thresholds (faithfulness 0.85, answer_relevancy 0.85, context_precision 0.80,
context_recall 0.80). A breach fails the test with a clear per-metric message. Make it
deterministic by stubbing the judge with fixed scores.

**Success criteria**
- Single `test_ragas_gate` function.
- Failing test prints the per-metric score + which threshold breached.
- Deterministic: the same RAG output yields the same assertion outcome.

In [ ]:
# Worked solution - a deterministic Ragas threshold gate. The judge is STUBBED with
# fixed aggregate scores so the assertion is bit-for-bit repeatable (no LLM, no network).
# In the customer repo the same test calls ragas.evaluate(...).to_pandas().mean() instead.

THRESHOLDS = {
    "faithfulness": 0.85,
    "answer_relevancy": 0.85,
    "context_precision": 0.80,   # the canonical 4.6 gate
    "context_recall": 0.80,
}

def stub_judge_scores(regression: bool = False) -> dict:
    """Fixed aggregate scores standing in for ragas.evaluate(...).to_pandas().mean()."""
    if regression:
        return {"faithfulness": 0.78, "answer_relevancy": 0.88,
                "context_precision": 0.82, "context_recall": 0.83}
    return {"faithfulness": 0.91, "answer_relevancy": 0.90,
            "context_precision": 0.86, "context_recall": 0.84}

def assert_gate(scores: dict, thresholds: dict) -> None:
    """Raise AssertionError naming every metric below its DPO-signed threshold."""
    breaches = [f"{m}={scores[m]:.2f} < {t:.2f}" for m, t in thresholds.items()
                if scores.get(m, 0.0) < t]
    assert not breaches, "Ragas CI gate FAILED: " + "; ".join(breaches)

def test_ragas_gate(regression: bool = False) -> None:
    assert_gate(stub_judge_scores(regression=regression), THRESHOLDS)

# clean run passes
test_ragas_gate(regression=False)
print("clean run: PASS (gate green)")

# a prompt regression trips faithfulness and the message names the breached metric
try:
    test_ragas_gate(regression=True)
    print("regression run: PASS (unexpected)")
except AssertionError as e:
    print("regression run: FAIL as expected ->", e)

---
## Exercise 4 - GitHub Actions workflow + PR comment  (Medium, 5 pts)

**Task.** Add a `.github/workflows/rag-ci.yml` that runs the Ragas gate on every PR. On failure,
post a comment to the PR with the breached metrics using `actions/github-script`. Keep the
evaluator LLM current (`gpt-5.6` or `claude-sonnet-5` as judge).

**Success criteria**
- Workflow triggered on `pull_request`.
- Secrets used for `OPENAI_API_KEY` + `ANTHROPIC_API_KEY`.
- PR comment lists the metric scores + which thresholds breached.
- A clean run marks the PR check green.

**Worked solution** (YAML - reference, not an executable cell):

```yaml
# .github/workflows/rag-ci.yml
name: rag-ci
on: [pull_request]
jobs:
  ragas-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.12' }
      - run: pip install -e '.[eval]'
      - id: gate
        continue-on-error: true
        run: pytest tests/test_ragas_gate.py --tb=short | tee gate.txt
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
      - if: steps.gate.outcome == 'failure'
        uses: actions/github-script@v7
        with:
          script: |
            const fs = require('fs');
            const log = fs.readFileSync('gate.txt', 'utf8').slice(-1500);
            await github.rest.issues.createComment({
              owner: context.repo.owner,
              repo: context.repo.repo,
              issue_number: context.issue.number,
              body: '## Ragas CI gate failed\n```\n' + log + '\n```',
            });
            core.setFailed('Ragas gate breached - see PR comment.');
```


---
## Exercise 5 - Customer scorecard generator - 1-page PDF/HTML  (Hard, 5 pts)

**Task.** Build `scorecard(scores, customer_name, period, ...)` that emits a 1-page HTML
(or PDF via `weasyprint`): 4 metric cards with current value + delta-from-last-quarter + a
traffic light vs the gate, a per-metric trend sparkline, an auto-generated commentary, and the
customer name + period in the header.

**Success criteria**
- Signature accepts an arbitrary customer name and period.
- All 4 metrics render with traffic-light colours (green at/above target, amber within ~5 points,
  rose below).
- Trend sparkline (inline SVG is fine).
- Commentary auto-generated from the score deltas.

In [ ]:
# Worked solution - scorecard(scores, customer_name, period, deltas, trends): a 1-page
# HTML report with per-metric cards (value + delta-from-last-quarter + traffic light), an
# inline-SVG trend sparkline, and auto-generated commentary. Returns an HTML string; write
# it to disk or render inline. No external libraries (weasyprint would turn it into a PDF).

GATE = {"faithfulness": 0.85, "answer_relevancy": 0.85,
        "context_precision": 0.80, "context_recall": 0.80}

def _light(value: float, gate: float) -> str:
    if value >= gate:
        return "#0d9488"          # green - at/above the gate
    if value >= gate - 0.05:
        return "#f59e0b"          # amber - within ~5 points
    return "#e11d48"              # rose - below

def _sparkline(trend: list, gate: float, w: int = 120, h: int = 28) -> str:
    lo, hi = min(trend + [gate]), max(trend + [gate])
    span = (hi - lo) or 1.0
    step = w / max(1, len(trend) - 1)
    pts = " ".join(f"{i*step:.1f},{h - (v-lo)/span*h:.1f}" for i, v in enumerate(trend))
    gy = h - (gate - lo) / span * h
    return (f'<svg width="{w}" height="{h}" viewBox="0 0 {w} {h}">'
            f'<line x1="0" y1="{gy:.1f}" x2="{w}" y2="{gy:.1f}" stroke="#94a3b8" '
            f'stroke-dasharray="3 3" stroke-width="1"/>'
            f'<polyline points="{pts}" fill="none" stroke="#0891b2" stroke-width="2"/></svg>')

def _commentary(scores: dict, deltas: dict) -> str:
    breached = [m for m in scores if scores[m] < GATE[m]]
    improved = [m for m, d in deltas.items() if d > 0.005]
    dropped = [m for m, d in deltas.items() if d < -0.005]
    parts = ["All four metrics clear the gate." if not breached
             else "Below gate: " + ", ".join(breached) + " - merge blocked."]
    if improved:
        parts.append("Up vs last quarter: " + ", ".join(improved) + ".")
    if dropped:
        parts.append("Down vs last quarter: " + ", ".join(dropped) + " - watch for drift.")
    return " ".join(parts)

def scorecard(scores: dict, customer_name: str, period: str,
              deltas: dict, trends: dict) -> str:
    cards = []
    for m, v in scores.items():
        colour = _light(v, GATE[m])
        d = deltas.get(m, 0.0)
        cards.append(
            f'<div style="border:1px solid #e7e5e4;border-top:4px solid {colour};'
            f'border-radius:10px;padding:12px 14px;min-width:150px">'
            f'<div style="font-size:11px;text-transform:uppercase;color:#64748b">{m}</div>'
            f'<div style="font-family:JetBrains Mono,monospace;font-size:24px;color:{colour}">{v:.2f}</div>'
            f'<div style="font-size:12px;color:#475569">{d:+.2f} vs Q-1, gate {GATE[m]:.2f}</div>'
            f'<div style="margin-top:6px">{_sparkline(trends[m], GATE[m])}</div></div>')
    return (f'<div style="font-family:DM Sans,sans-serif;max-width:720px;background:#fafaf9;'
            f'color:#0f1729;padding:22px;border:1px solid #e7e5e4;border-radius:10px">'
            f'<h2 style="font-family:Playfair Display,serif;margin:0">Ragas Scorecard</h2>'
            f'<p style="color:#334155;margin:4px 0 16px">Customer: <strong>{customer_name}</strong>'
            f', period <strong>{period}</strong>, region us-east1</p>'
            f'<div style="display:flex;gap:12px;flex-wrap:wrap">{"".join(cards)}</div>'
            f'<p style="margin-top:16px;color:#334155">{_commentary(scores, deltas)}</p></div>')

scores = {"faithfulness": 0.91, "answer_relevancy": 0.93,
          "context_precision": 0.86, "context_recall": 0.79}
deltas = {"faithfulness": 0.04, "answer_relevancy": 0.01,
          "context_precision": -0.02, "context_recall": -0.06}
trends = {"faithfulness": [0.86, 0.88, 0.91], "answer_relevancy": [0.90, 0.92, 0.93],
          "context_precision": [0.88, 0.87, 0.86], "context_recall": [0.85, 0.83, 0.79]}

report = scorecard(scores, customer_name="AcmeCorp EU-bank", period="Q3 2026",
                   deltas=deltas, trends=trends)
with open("scorecard-acmecorp.html", "w", encoding="utf-8") as f:
    f.write(report)
print("scorecard-acmecorp.html written:", len(report), "chars")
print("context_recall 0.79 < 0.80 gate -> its card renders rose; commentary flags the drop.")

---
## Exercise 6 - 50-question golden + Ragas eval + CI gate for AcmeCorp's EU-bank RAG  (Graded, 5 pts + 1 bonus)

**Task (Module 4 composite).** Submit: (1) a 50-row SME-style golden set in
`eval/golden.jsonl` mixing citation / paraphrase / multi-hop / non-English; (2) a Ragas 0.4.x
scorecard on a runnable RAG with all 4 metrics and a 3-run trend across at least 3 prompt/model
iterations; (3) a CI-gate YAML proven to fail on a deliberate regression, hardening to
faithfulness 0.90 for the enterprise-tier renewal; (4) a customer-facing 1-page report from the
Ex5 scorecard. **+1 bonus:** trend tracking across 3+ runs with delta annotations.

The worked cell below runs the composite at notebook scale (deterministic offline aggregates).

In [ ]:
# Worked solution - the Module 4 composite, at runnable scale. Builds a 50-row golden
# programmatically (the graded submission is hand-authored SME rows), runs the gate across
# 3 prompt/model iterations to show a trend, hardens faithfulness to 0.90 for the
# enterprise-tier renewal path, and emits the customer report from the Ex5 scorecard.
import json, os

# 1) 50-row golden set (demo generator; real submission is SME-authored, mixed types).
TYPES = ["citation", "paraphrase", "multi_hop", "non_english"]
LANGS = {"citation": "en", "paraphrase": "en", "multi_hop": "en", "non_english": "fr"}
golden50 = []
for i in range(50):
    t = TYPES[i % len(TYPES)]
    golden50.append({
        "user_input": f"[{t}] AcmeCorp EU-bank policy question #{i + 1}",
        "reference": "SME-authored ground-truth citing the governing GDPR / EU AI Act clause.",
        "reference_contexts": ["AcmeCorp policy chunk id " + str(i + 1)],
        "metadata": {"type": t, "lang": LANGS[t]},
    })
os.makedirs("eval", exist_ok=True)
with open("eval/golden.jsonl", "w", encoding="utf-8") as f:
    for row in golden50:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("eval/golden.jsonl:", len(golden50), "rows")

# 2) Three eval iterations (prompt/model tweaks) - deterministic offline aggregates.
ITERATIONS = {
    "v1 baseline prompt":  {"faithfulness": 0.81, "answer_relevancy": 0.88, "context_precision": 0.83, "context_recall": 0.80},
    "v2 add citations":    {"faithfulness": 0.87, "answer_relevancy": 0.90, "context_precision": 0.85, "context_recall": 0.82},
    "v3 rerank + rewrite": {"faithfulness": 0.92, "answer_relevancy": 0.93, "context_precision": 0.88, "context_recall": 0.84},
}
STANDARD_GATE = {"faithfulness": 0.85, "answer_relevancy": 0.85, "context_precision": 0.80, "context_recall": 0.80}
RENEWAL_GATE = {**STANDARD_GATE, "faithfulness": 0.90}   # hardens for the enterprise tier

def gate_report(scores: dict, gate: dict) -> str:
    breaches = [f"{m} {scores[m]:.2f}<{t:.2f}" for m, t in gate.items() if scores[m] < t]
    return "PASS" if not breaches else "FAIL: " + ", ".join(breaches)

print("\nTrend across 3 iterations (standard gate / renewal gate):")
prev = None
for name, s in ITERATIONS.items():
    delta = "" if prev is None else "  d_faith=%+.2f" % (s["faithfulness"] - prev)
    print(f"  {name:22s} std={gate_report(s, STANDARD_GATE):5s}  "
          f"renewal={gate_report(s, RENEWAL_GATE)}{delta}")
    prev = s["faithfulness"]
print("\nv1 fails the standard gate; v2 clears standard but fails the 0.90 renewal gate;")
print("only v3 clears the enterprise-tier renewal. That is the regression story for the PR.")
print("(Ex5 scorecard() renders the customer-facing 1-pager from the v3 aggregates.)")

---
## Exercise 7 - Judge-noise and the flaky gate  (Hard, 5 pts)

**Task.** An LLM-as-judge metric is a random variable. Given 5 repeated runs of the same
20-question eval for `main` and a byte-identical `candidate` PR, quantify the noise (per-run mean,
std, 95% bootstrap CI), watch the absolute 0.85 gate flip on identical code, then replace it with
a paired delta-vs-main gate (`FAIL iff delta < -0.03`) that cancels the shared judge noise. Runs
offline - the scores are inline.

**Success criteria**
- Variance table with per-run mean / std / bootstrap CI.
- A clear demo that the absolute 0.85 gate flips on identical code.
- A working paired delta-vs-main gate that passes all 5 runs.

In [ ]:
import numpy as np

# 5 repeated eval runs of the SAME 20-question golden set, SAME code, SAME model.
# Only the LLM judge's sampling noise changed between runs.
MAIN_RUNS = [
    [1.0]*14 + [0.5, 0.5, 0.5, 0.5, 0.3, 0.3],
    [1.0]*15 + [0.5, 0.5, 0.5, 0.5, 0.4],
    [1.0]*15 + [0.5, 0.5, 0.5, 0.25, 0.25],
    [1.0]*15 + [0.5, 0.5, 0.5, 0.5, 0.2],
    [1.0]*14 + [0.5, 0.5, 0.5, 0.5, 0.5, 0.3],
]
# The candidate PR is byte-identical to main, scored in the SAME judge batch (correlated noise).
CANDIDATE_RUNS = [
    [1.0]*14 + [0.5, 0.5, 0.5, 0.5, 0.3, 0.4],
    [1.0]*15 + [0.5, 0.5, 0.5, 0.5, 0.44],
    [1.0]*15 + [0.46, 0.5, 0.5, 0.25, 0.25],
    [1.0]*15 + [0.5, 0.5, 0.5, 0.5, 0.3],
    [1.0]*14 + [0.5, 0.5, 0.5, 0.5, 0.5, 0.34],
]
GATE = 0.85   # the naive absolute faithfulness gate

def boot_ci(run, n=10000, seed=42):
    rng = np.random.default_rng(seed)
    arr = np.array(run)
    means = [rng.choice(arr, size=len(arr), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

# TODO 1 (done): per-run mean, std, and the 95% bootstrap CI -> variance table.
print(f'{"run":>3} {"main":>6} {"cand":>6} {"c_std":>6} {"ci_lo":>6} {"ci_hi":>6}')
for i in range(5):
    m, c = float(np.mean(MAIN_RUNS[i])), float(np.mean(CANDIDATE_RUNS[i]))
    lo, hi = boot_ci(CANDIDATE_RUNS[i])
    print(f'{i+1:>3} {m:>6.3f} {c:>6.3f} {float(np.std(CANDIDATE_RUNS[i])):>6.3f} {lo:>6.3f} {hi:>6.3f}')

# TODO 2 (done): apply GATE to each CANDIDATE run mean -> watch the flip on identical code.
abs_verdicts = ["PASS" if np.mean(r) >= GATE else "FAIL" for r in CANDIDATE_RUNS]
print("\nabsolute gate (candidate mean >= 0.85):", abs_verdicts)

# TODO 3 (done): paired gate - delta = mean(candidate) - mean(main); FAIL iff delta < -0.03.
print("paired delta gate (vs main, tolerance -0.03):")
for i in range(5):
    d = float(np.mean(CANDIDATE_RUNS[i]) - np.mean(MAIN_RUNS[i]))
    print(f'  run {i+1}: delta={d:+.3f} -> {"PASS" if d >= -0.03 else "FAIL"}')
print("The shared judge noise cancels in the paired delta, so all 5 runs pass.")

---
## Exercise 8 - Calibrate the judge to the SME  (Hard, 5 pts)

**Task.** Before trusting an automatic faithfulness score as a gate, prove it agrees with
the human it stands in for (Shankar et al. 2024, arxiv.org/abs/2404.12272). Given 30 golden rows
an SME hand-labelled supported / unsupported, each paired with the Ragas faithfulness score:
measure Pearson correlation, sweep the threshold computing Cohen's kappa + the 2x2 confusion at
each, then DERIVE the gate as argmax kappa instead of guessing 0.85. Runs offline.

**Success criteria**
- A correct Cohen's kappa (chance-corrected agreement, not raw accuracy).
- The full 2x2 confusion matrix and correlation.
- A threshold DERIVED from the kappa sweep (argmax near 0.82), not hand-picked.

In [ ]:
import numpy as np

# 30 golden rows an SME hand-labelled: (ragas_faithfulness_score, sme_label)
# sme_label: 1 = SME agrees the answer IS supported by its cited context, 0 = not.
GOLDEN = [
    (0.41, 0), (0.53, 0), (0.57, 0), (0.60, 0), (0.64, 0), (0.67, 0), (0.69, 0), (0.72, 0),
    (0.74, 0), (0.77, 0), (0.79, 0), (0.81, 0),
    (0.88, 0),                       # confident hallucination: high judge score, SME rejected
    (0.825, 1), (0.845, 1),          # borderline-correct: low judge score, SME accepted
    (0.85, 1), (0.86, 1), (0.87, 1), (0.89, 1), (0.90, 1), (0.91, 1), (0.92, 1), (0.93, 1),
    (0.94, 1), (0.95, 1), (0.96, 1), (0.97, 1), (0.98, 1), (0.99, 1), (1.00, 1),
]

def cohen_kappa(pred, truth):
    pred, truth = np.array(pred), np.array(truth)
    po = (pred == truth).mean()
    pe = sum((pred == c).mean() * (truth == c).mean() for c in (0, 1))
    return (po - pe) / (1 - pe)

def confusion(pred, truth):
    pred, truth = np.array(pred), np.array(truth)
    tp = int(((pred == 1) & (truth == 1)).sum()); fp = int(((pred == 1) & (truth == 0)).sum())
    fn = int(((pred == 0) & (truth == 1)).sum()); tn = int(((pred == 0) & (truth == 0)).sum())
    return tp, fp, fn, tn

scores = np.array([s for s, _ in GOLDEN])
labels = np.array([y for _, y in GOLDEN])

# TODO 1 (done): correlation between judge score and SME label.
r = float(np.corrcoef(scores, labels)[0, 1])
print(f"Pearson r (judge score vs SME label): {r:.2f}")

# TODO 2 (done): sweep t, compute kappa + confusion at each; TODO 3: argmax kappa.
best_t, best_k = 0.85, -1.0
for t in np.arange(0.70, 1.001, 0.01):
    k = cohen_kappa((scores >= t).astype(int), labels)
    if k > best_k:
        best_k, best_t = k, float(t)

for t in (0.85, round(best_t, 2)):
    pred = (scores >= t).astype(int)
    tp, fp, fn, tn = confusion(pred, labels)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    tag = "naive" if t == 0.85 else "calibrated"
    print(f"  {tag:10s} t={t:.2f}: kappa={cohen_kappa(pred, labels):.2f}  "
          f"TP={tp} FP={fp} FN={fn} TN={tn}  precision={prec:.2f} recall={rec:.2f}")
print(f"\nCalibrated gate = argmax kappa = {best_t:.2f} (not the guessed 0.85). "
      "Re-label and recalibrate as SME criteria drift (Shankar et al. 2024).")

---
## Grading summary - 40 pts total

Pass = 28/40. Distinction = 36/40. Reviewers watch for these common pitfalls:

- **FDE-authored golden set without SME framing** - the answers must read like a compliance
  officer wrote them. Auto-deduct 3 pts.
- **Only measuring faithfulness** - both faithfulness AND context precision are required gates.
  Auto-deduct 2 pts.
- **Single-run absolute gate with no variance analysis (Ex7)** - a gate you never stress-tested
  for judge noise will flake in CI. Auto-deduct 2 pts.
- **Guessed threshold instead of a calibrated one (Ex8)** - a gate threshold must be derived from
  SME agreement, not picked because 0.85 sounds right. Auto-deduct 2 pts.
- **Mocked Ragas without honesty disclosure** - if you stub the LLM judge, say so loudly. Hidden
  mocks: auto-deduct 2 pts.
- **Dishonest measurement** - reported scores must match printed eval output. Mismatch:
  auto-deduct 5 pts.

**Submission checklist**
- [ ] Ex1 per-row 0.4.x scorecard + which-metric-flagged-which commentary
- [ ] Ex2 `golden.jsonl` - 10 rows, SME-style language, >= 1 non-English
- [ ] Ex3 deterministic `test_ragas_gate` with a clear per-metric failure message
- [ ] Ex4 `.github/workflows/rag-ci.yml` that comments on failure
- [ ] Ex5 `scorecard(...)` - 4 traffic-light cards + sparkline + auto commentary, 1 page
- [ ] Ex6 50-row golden + 3-iteration trend + renewal-tier 0.90 gate + customer report
- [ ] Ex7 variance table + absolute-gate flip + paired delta gate
- [ ] Ex8 Pearson r + kappa sweep + calibrated argmax threshold (not 0.85)

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback
is returned within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked
notebook and the rubric to self-assess right away.
